In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

In [ ]:
#Cargamos archivos
df_puerto = pd.read_csv('datosPuerto.csv')
df_puerto.head()

In [ ]:
#Filtramos las partidas arancelarias 
df_puerto['capitulo'] = df_puerto['PARTIDA_ARANCELARIA'].astype(str).str[:2]

In [ ]:
#FIltramos para que solo sean importanciones y exportaciones a Paises bajos
df_puerto = df_puerto[((df_puerto['PAIS_ORIGEN'] == 'NL') & (df_puerto['PAIS_FINAL'] == 'ES')) | ((df_puerto['PAIS_ORIGEN'] == 'ES') & (df_puerto['PAIS_FINAL'] == 'NL'))]

In [ ]:
#Agrupamos los kilos por pais y capitulo, y los ordenamos de mayor a menor para ver los capitulos mas importantes
ranking_puerto = df_puerto.groupby(['PAIS_ORIGEN', 'capitulo'])['PESO_KILOS'].sum().reset_index()
ranking_puerto = ranking_puerto.sort_values(by='PESO_KILOS', ascending=False)

In [ ]:
ranking_puerto

In [ ]:
#codigos_arancelarios_capitulos.pdf:

#CAPÍTULO 27:	Combustibles minerales, aceites minerales y productos de su destilación, materias bituminosas, ceras minerales.
#CAPÍTULO 72:	Fundición, hierro y acero
#CAPÍTULO 25:	Sal, azufre, tierras y piedras, yesos, cales y cementos. 
#CAPITULO 29:   Productos químicos orgánicos.

In [ ]:
#Cargamos los datos para hacer la importancia del puerto de bilbao en españa pero solo con los capitulos importantes
comex = pd.read_csv("DataComex-import-export-españa.csv", sep=";", decimal=",")
puerto = pd.read_csv("datosPuerto.csv", decimal=",")

In [ ]:
#Cambiamos la fecha a datetimo y ponemos solo el año
puerto['FECHA'] = pd.to_datetime(puerto['FECHA'])
puerto['FECHA'] = puerto['FECHA'].dt.year

In [ ]:
puerto

In [ ]:
#Filtramos para que el periodo de años sea del 2009 al 2019(es hasta 2019 poprque el dataset de euskadi solo llega hasta 2019)
comex_filtrado = comex[(comex["pais_nombre"] == "Países Bajos") & (comex["periodo_anio"] >= 2009) & (comex["periodo_anio"] <= 2019)]

In [ ]:
#Filtramos para que solo sea el puerto respecto a paises bajos
puerto_paisesbajos = puerto[((puerto['PAIS_ORIGEN'] == 'NL') & (puerto['PAIS_FINAL'] == 'ES')) | ((puerto['PAIS_ORIGEN'] == 'ES') & (puerto['PAIS_FINAL'] == 'NL'))]

In [ ]:
comex_filtrado

In [ ]:
#Filtrammos de nuevo para que solo salgan los capitulos
puerto_paisesbajos['capitulo'] = puerto_paisesbajos['PARTIDA_ARANCELARIA'].astype(str).str[:2]

In [ ]:
puerto_paisesbajos

In [ ]:
#Ponemos el numero de capituloen int
puerto_paisesbajos["capitulo"]=puerto_paisesbajos["capitulo"].astype(int)

In [ ]:
#Filtramos para que solo sean las exportyaciones y importaciones de los capitulos mencionados
capitulos = [25, 27, 29, 72]
comex_filtrado = comex_filtrado[comex_filtrado["taric"].isin(capitulos)]

puerto_paisesbajos = puerto_paisesbajos[puerto_paisesbajos["capitulo"].isin(capitulos)]

In [ ]:
puerto_paisesbajos

In [ ]:
#Agrupamos los kilos por años y separamos en exportaciones y importaciones
total_exp_es = (comex_filtrado[comex_filtrado["flujo_nombre"] == "Exportación"].groupby("periodo_anio")["kilos"].sum())

total_imp_es = (comex_filtrado[comex_filtrado["flujo_nombre"] == "Importación"].groupby("periodo_anio")["kilos"].sum())

In [ ]:
#FIltramos solo paises bajos(De nuevo por si acaso)
exp_puerto = puerto_paisesbajos[(puerto_paisesbajos["PAIS_ORIGEN"] == "ES") & (puerto_paisesbajos["PAIS_FINAL"] == "NL")]
imp_puerto = puerto_paisesbajos[(puerto_paisesbajos["PAIS_ORIGEN"] == "NL") & (puerto_paisesbajos["PAIS_FINAL"] == "ES")]

In [ ]:
#Agrupamos los kilos por años
imp_puerto = imp_puerto.groupby("FECHA")["PESO_KILOS"].sum()
exp_puerto =exp_puerto.groupby("FECHA")["PESO_KILOS"].sum()

In [ ]:
#Filtramos solo los años de 2009 a 2019
exp_puerto=exp_puerto.loc[2009:2019]
imp_puerto=imp_puerto.loc[2009:2019]

In [ ]:
porcentaje_exp = (exp_puerto / total_exp_es) * 100
porcentaje_exp

In [ ]:
porcentaje_imp = (imp_puerto / total_imp_es) * 100
porcentaje_imp

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
años = [2009, 2010, 2011, 2012, 2013,2014, 2015, 2016, 2017, 2018, 2019]

porcentaje_exp = [20.628531, 18.936257, 40.507670, 18.773306, 38.844094,30.999656, 14.341037, 19.452463, 17.234240, 17.152005, 22.615871]

porcentaje_imp = [19.220859, 26.099254, 17.913981, 21.611057, 21.301688,23.508977, 13.392031, 15.073102, 18.368581, 13.416463, 16.188671]

plt.figure(figsize=(10,5))
sns.lineplot(x=años, y=porcentaje_exp, marker="o", label="porcentaje_exp")
sns.lineplot(x=años, y=porcentaje_imp, marker="o", label="porcentaje_imp")
plt.title("porcentaje_exp vs porcentaje_imp capítulos más importantes")
plt.xlabel("Año")
plt.ylabel("Porcentaje")
plt.grid(True)
plt.legend()
plt.show()